# Beyond least squares

## What is maximum likelihood estimation?

In the [previous chapter](2:regression), you saw that a model for linear regression with a single predictor takes the form:

$$y_i = \beta_0 + \beta_1 x_i + \epsilon_i$$

Where:
  * $y_i$ is an observation of the outcome.
  * $x_i$ is an observation of the predictor.
  * $\beta_0$ is the intercept.
  * $\beta_1$ is the slope.
  * $\epsilon_i$ is the error.

And that ordinary least squares provides a closed-form solution to the estimation of the slope and the intercept based on minimizing the sum of the squares of the residuals, so that you get:

$$\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i$$

Ordinary least squares only provides a single, best estimate of the outcome, which is called a point estimate or a point prediction. Quantifying uncertainty requires an extra step, either by assuming normality of the error or by bootstrapping.

But it is also possible to reframe the linear model as a probabilistic model:

$$y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_i, \sigma_\epsilon^2)$$

So you can directly get a distribution estimate of the outcome, here parameterized as a normal distribution of mean $\beta_0 + \beta_1 x_i$ and variance $\sigma_\epsilon^2$ (so it assumes that the error is normally distributed). In this model, you have three parameters to estimate: $\beta_0$, $\beta_1$, and $\sigma_\epsilon^2$. For that, you need to move beyond residuals and take a probabilistic perspective on fitting.

```{tip}

Multiple linear regression can be reframed in the exact same way. Instead of:

$$y_i = \beta_0 + \beta_1 x_{1,i} + \cdots + \beta_m x_{m,i} + \epsilon_i$$

which has $m$ predictors, you get:

$$y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_{1,i} + \cdots + \beta_m x_{m,i}, \sigma_\epsilon^2)$$
```

(2:beyond_least_squares:likelihood)=
### Likelihood

At the core of probabilistic parameter estimation lies the concept of likelihood. The idea is to quantify how likely some parameter values for a model of probability distribution are given the data that we have observed. In practice, the likelihood is the [probability density](1:continrandom:pdf) of the data given the parameters and predictors. So, for a normal distribution and a single observation, you get:

$$
    \begin{align}
        \mathcal{L}(\hat{\beta}_0, \hat{\beta}_1, \hat{\sigma}_\epsilon^2) &= f(y_i|\hat{\beta}_0 + \hat{\beta_1} x_i, \hat{\sigma}_\epsilon^2) \\
         &= \frac{1}{\sqrt{2\pi\hat{\sigma}_\epsilon^2}}e^{\displaystyle-\frac{y_i - \hat{\beta}_0 - \hat{\beta_1} x_i}{2\hat{\sigma}_\epsilon^2}}
    \end{align}
$$

When we have a model of probability distribution, the likelihood is often called likelihood function, similarly to the probability density function. In the function above, you can see that the likelihood increases when $\hat{\beta}_0 - \hat{\beta_1} x_i$ gets closer to $y_i$ or when $\hat{\sigma}_\epsilon$ gets closer to 0. So, while lower residuals mean a better fit to the data, a higher likelihood means a better fit to the data.

With $n$ observations, the likelihood becomes the joint probability density over the data. To compute it, we usually assume that the data are [independent](Indep), so that the likelihood function becomes:

$$
    \begin{align}
        \mathcal{L}(\hat{\beta}_0, \hat{\beta}_1, \hat{\sigma}_\epsilon^2) &= \prod_{i=1}^n f(y_i|\hat{\beta}_0 + \hat{\beta_1} x_i, \hat{\sigma}_\epsilon^2) \\
         &= \prod_{i=1}^n \frac{1}{\sqrt{2\pi\hat{\sigma}_\epsilon^2}}e^{\displaystyle-\frac{y_i - \hat{\beta}_0 - \hat{\beta_1} x_i}{2\hat{\sigma}_\epsilon^2}} \\
         &= \frac{1}{(2\pi\hat{\sigma}_\epsilon^2)^{n/2}}e^{\displaystyle-\frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta_1} x_i)^2}
    \end{align}
$$

The square operation inside an exponential is numerically costly, and it is prone to return either tiny or huge numbers, leading to numerical instability. So in practice we use the log likelihood:

$$
    \begin{align}
        \mathcal{L}\mathcal{L}(\hat{\beta}_0, \hat{\beta}_1, \hat{\sigma}_\epsilon^2) =& -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^{n} (y_i - \hat{\beta}_0 - \hat{\beta_1} x_i)^2 \\
          =& -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^{n} r_i^2 \\
    \end{align}
$$

Where $r_i$ is the residual for observation $i$ and $\log$ stands for the natural logarithm, i.e., the logarithm to the base $e$. Since the logarithm is a monotonic function, it preserves the original orderering of the likelihood, so that both lead to the same best parameter values.

### Maximum likelihood estimation

Finding estimates for $\beta_0$, $\beta_1$, and $\sigma^2$ using the log likelihood means solving the following maximization problem:

$$\underset{\hat{\beta}_0,\hat{\beta}_1,\hat{\sigma}_\epsilon^2}{\max} \mathcal{L}\mathcal{L}(\hat{\beta}_0,\hat{\beta}_1,\hat{\sigma}_\epsilon^2)$$

To find a closed-form solution to that problem, we need to solve:

$$\frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\beta}_0} = 0; \qquad \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\beta}_1} = 0; \qquad \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\sigma}_\epsilon^2} = 0$$

Let's start with the first one and find $\hat{\beta}_0$. First, we apply the linearity of differenciation and the chain rule:

$$
    \begin{align}
        \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\beta}_0} &= 0 - 0 - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n\frac{\partial r_i^2}{\partial r_i}\frac{\partial r_i}{\partial \hat{\beta}_0} \\
        &= - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n 2r_i \cdot (-1) \\
        &= \frac{1}{\hat{\sigma}_\epsilon^2}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i) \\
    \end{align}
$$

From there, remember that the sample mean for $x$ and $y$ are:

$$
    \begin{align}
        \overline{x} &= \frac{1}{n}\sum_{i=1}^n x_i \\
        \overline{y} &= \frac{1}{n}\sum_{i=1}^n y_i \\
    \end{align}
$$

And we can use those to remove the sum:

$$
    \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\beta}_0} = \frac{n}{\hat{\sigma}_\epsilon^2}(\overline{y} - \hat{\beta}_0 - \hat{\beta}_1 \overline{x}) = 0
$$

So that:

$$\hat{\beta}_0 = \overline{y} - \hat{\beta}_1 \overline{x}$$

Now we have the intercept, which depends on the slope. So let's turn to the second of our initial partial derivatives to find $\hat{\beta}_1$. First, we can substitute for $\hat{\beta}_0$ in the log likelihood:

$$
    \begin{align}
        \mathcal{L}\mathcal{L}(\hat{\beta}_0,\hat{\beta}_1,\hat{\sigma}_\epsilon^2) &= -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2 \\
         &= -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n (y_i - \overline{y} + \hat{\beta}_1 \overline{x} - \hat{\beta}_1 x_i)^2 \\
         &= -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n \left[(y_i - \overline{y}) - \hat{\beta}_1 (x_i - \overline{x})\right]^2 \\
    \end{align}
$$

Let's call the element within the sum $r'_i$:

$$r'_i = (y_i - \overline{y}) - \hat{\beta}_1 (x_i - \overline{x})$$

Again, we apply the linearity of differenciation and the chain rule:

$$
    \begin{align}
        \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\beta}_1} &= 0 - 0 - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n\frac{\partial {r'}_i^2}{\partial r'_i}\frac{\partial r'_i}{\partial \hat{\beta}_1} \\
        &= - \frac{1}{2\hat{\sigma}_\epsilon^2}\sum_{i=1}^n 2r'_i \cdot (\overline{x} - x_i) \\
        &= \frac{1}{\hat{\sigma}_\epsilon^2}\sum_{i=1}^n \left[(y_i - \overline{y}) - \hat{\beta}_1 (x_i - \overline{x})\right](x_i - \overline{x}) \\
        &= \frac{1}{\hat{\sigma}_\epsilon^2}\left(\sum_{i=1}^n (y_i - \overline{y})(x_i - \overline{x}) - \hat{\beta}_1\sum_{i=1}^n (x_i - \overline{x})^2 \right) \\
        &= 0 \\
    \end{align}
$$

So that:

$$
    \hat{\beta}_1 = \frac{\sum_{i=1}^n (y_i - \overline{y})(x_i - \overline{x})}{\sum_{i=1}^n (x_i - \overline{x})^2}
$$

Now we have the intercept and the slope, so let's finish with the variance of the error, where we simpy take the derivative of $\log{a}$ and $\frac{1}{a}$:

$$
    \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\sigma}_\epsilon^2} = 0 - \frac{n}{2\hat{\sigma}_\epsilon^2} - \frac{1}{2(\hat{\sigma}_\epsilon^2)^2}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2 = 0
$$

So that:

$$
    \hat{\sigma}_\epsilon^2 = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2
$$

And we now have the three formulas to get the maximum likelihood estimates for simple linear regression:

$$
    \begin{align}
        \hat{\beta}_1 &= \frac{\sum_{i=1}^n (y_i - \overline{y})(x_i - \overline{x})}{\sum_{i=1}^n (x_i - \overline{x})^2} \\
        \hat{\beta}_0 &= \overline{y} - \hat{\beta}_1 \overline{x} \\
        \hat{\sigma}_\epsilon^2 &= \frac{1}{n} \sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2 \\
    \end{align}
$$

The solutions for $\hat{\beta}_0$ and $\hat{\beta}_1$ are the same for ordinary least squares and maximum likelihood estimation when the error is normally distributed. When [estimating confidence intervals](2:uncertainty:parameters), we used for the variance:

$$
    \hat{\sigma}_\epsilon^2 = \frac{\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2}{n-2}
$$

Maximum likelihood estimation gave us the theoretical formula for $\hat{\sigma}_\epsilon^2$ but, just like the standard deviation, it can be biased when used on a small sample. Dividing by $n-2$ instead of $n$ reduces this bias, just like [Bessel's correction](1:descriptive_statistics:std) with the standard deviation.

```{admonition} Link to linear algebra
:class: tip

[Just like the sum of the squares of the residuals](2:regression:ols), the log likelihood can be written in matrix form that works for simple and multiple linear regression:

$$
    \mathcal{L}\mathcal{L}(\boldsymbol{\hat{\beta}}, \hat{\sigma}_\epsilon^2) = -\frac{n}{2}\log{2\pi} - \frac{n}{2}\log{\hat{\sigma}_\epsilon^2} - \frac{1}{2\hat{\sigma}_\epsilon^2}\left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)^T \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)    
$$

Again, the derivation with respect to $\boldsymbol{\hat{\beta}}$ follows similar steps to the derivation in ordinary least squares:

$$
    \begin{align}
        \frac{\partial \mathcal{L}\mathcal{L}}{\partial \boldsymbol{\hat{\beta}}} &= 0 - 0 - \frac{1}{2\hat{\sigma}_\epsilon^2}\frac{\partial}{\partial \boldsymbol{\hat{\beta}}}\left[ \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)^T \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \right] \\
         &= -\frac{1}{2\hat{\sigma}_\epsilon^2}\frac{\partial}{\partial \boldsymbol{\hat{\beta}}}\left( \boldsymbol{y}^T \boldsymbol{y} - 2\boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{y} + \boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}} \right) \\
         &= -\frac{1}{2\hat{\sigma}_\epsilon^2}\left(-2\boldsymbol{X}^T \boldsymbol{y} + 2\boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
         &= 0 \\
    \end{align}
$$

So that:

$$\boldsymbol{\hat{\beta}} = \left(\boldsymbol{X}^T \boldsymbol{X}\right)^{-1} \boldsymbol{X}^T \boldsymbol{y}$$

The derivation with respect to $\hat{\sigma}_\epsilon^2$ remains similar to the derivation of the simple linear case:

$$
    \frac{\partial \mathcal{L}\mathcal{L}}{\partial \hat{\sigma}_\epsilon^2} = 0 - \frac{n}{2\hat{\sigma}_\epsilon^2} - \frac{1}{2(\hat{\sigma}_\epsilon^2)^2}\left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)^T \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) = 0
$$

So that:

$$
    \hat{\sigma}_\epsilon^2 = \frac{1}{n} \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)^T \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)
$$
```

(2:beyond_least_squares:gradient_descent)=
## What is gradient descent?

Linear regression is a simple-enough problem that minimizing the sum of the squares of the residuals and maximizing the log likelihood have closed-form solutions. But what should you do when no closed-form solution exists?

This is where iterative optimization methods come into play. In general, optimization implies finding the minimum or maximum of a function as quickly and efficiently as possible. When following an iterative scheme, the idea is to start from an initial guess of parameter values, then to progressively update that guess to converge (hopefully) towards the best values.

### Objective function

The first key element of an optimization problem is the objective function. The objective function, also called the loss function or the cost function, is the function to minimize or maximize, so in our case some measure of fit to the data. The sum of the squares of the residuals and the log likelihood are two examples of objective functions.

In practice, most optimization problems are formulated as a minimization, so that the negative log likelihood is often used instead of the log likelihood. The sum of the squares of the residuals can lead quickly lead to huge values and numerical instability, so that the mean squared error is often used instead:

$$
    \begin{align}
        MSE(\hat{\beta}_0,\hat{\beta}_1) &= \frac{1}{n}\sum_{i=1}^n r_i^2 \\
         &= \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2 \\
         &= \frac{1}{n}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2 \\
    \end{align}
$$

### Gradient descent

The optimization algorithm is the second key element of an optimization problem. The optimization algorithm, also called the optimizer, is the algorithm to minimize or maximize the objective function. Here, we will focus on the simplest approach for parameter optimization: gradient descent.

Gradient descent can be used to minimize any differentiable function. To do so, it takes iterative steps in the direction opposite to the gradient based on the following update rule:

$$
    \hat{\beta}^{(t+1)}_j = \hat{\beta}^{(t)}_j - \eta \frac{\partial MSE}{\partial \hat{\beta}_j}
$$

Where:
  * $\hat{\beta}^{(t)}_j$ is the value of a parameter $\hat{\beta}_j$ at iteration $t$.
  * $\hat{\beta}^{(t+1)}_j$ is the value of the parameter after the update, so at iteration $t+1$.
  * $MSE$ is the mean squared error, but any other objective function can be used as long as it is differentiable.
  * $\eta$ is the step size, also called the learning rate, which controls how far the algorithm progresses between each iteration.

The initial values for the parameters ($t = 0$) can be randomly generated, set at a specific value (e.g., 0), or selected based on prior knowledge of the problem.

The step size is what is called in machine learning a hyperparameter, i.e., a parameter that is not estimated from the data during fitting, so that it must be user-defined prior to fitting. Tuning the step size is essential to the success of gradient descent:
  * A value too low means a slow convergence towards the minimum and an inefficient process.
  * A value too high means large jumps that can impede or even prevent convergence.

The number of iterations is another hyperparameter.

This process has some guarantees of convergence towards a global minimum when the objective function is [convex](https://en.wikipedia.org/wiki/Convex_function). If the objective function is not convex, gradient descent can get stuck in some local minimum, leading to a suboptimal solution. You will see later in you studies more advanced approaches to address those issues.

As mentioned already, gradient descent requires a differentiable objective function. In the case of the mean squared error and of a simple linear regression, we need $\displaystyle\frac{\partial MSE}{\partial \hat{\beta}_0}$ and $\displaystyle\frac{\partial MSE}{\partial \hat{\beta}_1}$. The derivation process is similar to finding the closed-form solutions of ordinary least squares and maximum likelihood estimation.

For $\hat{\beta}_0$, applying the linearity of differenciation and the chain rule leads to:

$$
    \begin{align}
        \frac{MSE}{\partial \hat{\beta}_0} &= \frac{\partial}{\partial \hat{\beta}_0}\frac{1}{n}\sum_{i=1}^n r_i^2 = \frac{1}{n}\sum_{i=1}^n\frac{\partial r_i^2}{\partial \hat{\beta}_0} \\
        &= \frac{1}{n}\sum_{i=1}^n\frac{\partial r_i^2}{\partial r_i}\frac{\partial r_i}{\partial \hat{\beta}_0} = \frac{1}{n}\sum_{i=1}^n 2r_i \cdot (-1) = -\frac{2}{n}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i) \\
    \end{align}
$$

For $\hat{\beta}_1$, applying the linearity of differenciation and the chain rule leads to:

$$
    \begin{align}
        \frac{MSE}{\partial \hat{\beta}_1} &= \frac{\partial}{\partial \hat{\beta}_1}\frac{1}{n}\sum_{i=1}^n r_i^2 = \frac{1}{n}\sum_{i=1}^n\frac{\partial r_i^2}{\partial \hat{\beta}_1} \\
        &= \frac{1}{n}\sum_{i=1}^n\frac{\partial r_i^2}{\partial r_i}\frac{\partial r_i}{\partial \hat{\beta}_1} = \frac{1}{n}\sum_{i=1}^n 2r_i \cdot (-x_i) = -\frac{2}{n}\sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)x_i \\
    \end{align}
$$

So when applying gradient descent the update rule becomes:

$$
    \begin{align}
        \hat{\beta}^{(t+1)}_0 &= \hat{\beta}^{(t)}_0 - \eta \frac{\partial MSE}{\partial \hat{\beta}_0} \qquad \text{with}\quad \frac{\partial MSE}{\partial \hat{\beta}_0} = -\frac{2}{n}\sum_{i=1}^n (y_i - \hat{\beta}^{(t)}_0 - \hat{\beta}^{(t)}_1 x_i) \\
        \hat{\beta}^{(t+1)}_1 &= \hat{\beta}^{(t)}_1 - \eta \frac{\partial MSE}{\partial \hat{\beta}_1} \qquad \text{with}\quad \frac{\partial MSE}{\partial \hat{\beta}_1} = -\frac{2}{n}\sum_{i=1}^n (y_i - \hat{\beta}^{(t)}_0 - \hat{\beta}^{(t)}_1 x_i)x_i \\
    \end{align}
$$

In the case of the (negative) log likelihood, you need to add a third term to the update rule for $\hat{\sigma}_\epsilon^2$.

```{admonition} Link to linear algebra
:class: tip

Following the example of the sum of the squares of the residuals, the mean squared error can be written in matrix form:

$$
    \begin{align}
        MSE(\boldsymbol{\hat{\beta}}) &= \frac{1}{n}\boldsymbol{r}^T \boldsymbol{r} \\
         &= \frac{1}{n}\left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)^T \left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
         &= \frac{1}{n}\left(\boldsymbol{y}^T \boldsymbol{y} - \boldsymbol{y}^T \boldsymbol{X}\boldsymbol{\hat{\beta}} - \boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{y} + \boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
         &= \frac{1}{n}\left(\boldsymbol{y}^T \boldsymbol{y} - 2\boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{y} + \boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
    \end{align}
$$

So that:

$$
    \begin{align}
        \frac{\partial MSE}{\partial \boldsymbol{\hat{\beta}}} &= \frac{\partial}{\partial \boldsymbol{\hat{\beta}}}\left[\frac{1}{n}\left( \boldsymbol{y}^T \boldsymbol{y} - 2\boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{y} + \boldsymbol{\hat{\beta}}^T \boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}} \right)\right] \\
         &= \frac{1}{n}\left(-2\boldsymbol{X}^T \boldsymbol{y} + 2\boldsymbol{X}^T \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
         &= -\frac{2}{n}\boldsymbol{X}^T\left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right) \\
    \end{align}
$$

And the update rule becomes:

$$
    \boldsymbol{\hat{\beta}^{(t+1)}} = \boldsymbol{\hat{\beta}^{(t)}} - \eta\frac{\partial MSE}{\partial \boldsymbol{\hat{\beta}}} \qquad \text{with}\quad \frac{\partial MSE}{\partial \boldsymbol{\hat{\beta}}} = -\frac{2}{n}\boldsymbol{X}^T\left(\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\hat{\beta}}\right)
$$

Here we directly update all the parameters with a single formula, in the simple and multiple linear regression cases.
```

## Activity: Implementing gradient descent

Now let's have a look at applying gradient descent with NumPy. For that, make sure to start the interactive Python environment by clicking on {fa}`rocket` {fa}`arrow-right-long` {guilabel}`Live Code` at the top of this page (then wait until the Python interaction is ready).

First, you need to import some packages:

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

Then, load the data into two NumPy arrays (data from [10.5066/P9SHOOH0](https://www.sciencebase.gov/catalog/item/631405eed34e36012efa3505)):

In [ ]:
width = np.loadtxt(Path.cwd().parent/'../data/ohio_streams_usgs_2005_5153.csv', delimiter=',', skiprows=1, usecols=0)
discharge = np.loadtxt(Path.cwd().parent/'../data/ohio_streams_usgs_2005_5153.csv', delimiter=',', skiprows=1, usecols=1)

`width` contains the bankfull width, the predictor, and `discharge` contains the bankfull discharge, the outcome.

In a [previous activity](2:regression:activity), you found the estimates for the intercept $\beta_0$ and the slope $\beta_1$ using ordinary least squares, which were:

$$
    \begin{align}
        \hat{\beta}_{0|\text{OLS}} &= -8.4439 \\
        \hat{\beta}_{1|\text{OLS}} &= 1.9824 \\
    \end{align}
$$

Now let's see if you can find the same values with gradient descent:
  1. Using NumPy, implement gradient descent with the mean squared error as objective function to compute the slope and intercept.
  2. Using Matplotlib, plot the evolution of the parameter values over the different iterations.
  3. How does the step size and the number of iterations affect that evolution?

In [ ]:
# Your answer here.